In [1]:
#Import necessary libraries
import pandas as pd
import requests
import json

# Url to the JSON data containing competition information
competitions_url = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json"
# Read the JSON data from the URL into a pandas DataFrame
competitions = pd.read_json(competitions_url)
# Display the first 15 rows of the DataFrame with available tournments
print (competitions[['competition_id', 'season_id', 'competition_name', 'season_name']].head(15))

# Get the events from a specific game
match_id = 8658
events_url = f"https://raw.githubusercontent.com/hudl/open-data/refs/heads/master/data/events/{match_id}.json"


raw = requests.get(events_url).json()
events = pd.json_normalize(raw)

print(events.shape)

# Keep only the passes and extract the x and y coordinates from the location column
passes = events[events['type.name'] == 'Pass'].copy()
passes['x'] = passes["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
passes['y'] = passes["location"].apply(lambda x: x[1] if isinstance(x, list) else None)
passes[['minute', 'player.name', 'x', 'y']].head(10)



print(passes.columns.tolist())

    competition_id  season_id        competition_name season_name
0                9        281           1. Bundesliga   2023/2024
1                9         27           1. Bundesliga   2015/2016
2             1267        107  African Cup of Nations        2023
3               16          4        Champions League   2018/2019
4               16          1        Champions League   2017/2018
5               16          2        Champions League   2016/2017
6               16         27        Champions League   2015/2016
7               16         26        Champions League   2014/2015
8               16         25        Champions League   2013/2014
9               16         24        Champions League   2012/2013
10              16         23        Champions League   2011/2012
11              16         22        Champions League   2010/2011
12              16         21        Champions League   2009/2010
13              16         41        Champions League   2008/2009
14        

In [2]:
# Tells Python to look at modules inn \src folder
import sys
sys.path.append('../src')

# Import files already created in the src folder
from etl.extract import get_competitions, get_matches, get_events
from etl.transform import get_passes, build_player_match_stats
from etl.load import create_tables, save_to_db

# Create tables (for the first time)
create_tables(reset=True)

# Bring events from a specific match
match_id = 8658
event = get_events(match_id)

# Calculate stats on a specific player
player_name = "Luka Modrić"
stats = build_player_match_stats(event, player_name, match_id)

# Save the passes to a database
save_to_db(stats, "Luka_Modrić_Match_Stats")

# Verify
stats

Tables created successfully.
Saved: 1 rows in table 'Luka_Modrić_Match_Stats'


,match_id,player_name,total_passes,total_progressive_passes,total_won_duels,total_succesfull_interceptions
0,8658,Luka Modrić,77,20,0,0


In [3]:
# Import needed libraries
import pandas as pd
from sqlalchemy import create_engine

# Create a connection to the SQLite database
Engine = create_engine('sqlite:///../data/processed/football_scouting.db')

# Check if the data was saved correctly by querying the database
check = pd.read_sql('SELECT * FROM Passes_match_8658 LIMIT 5', Engine)
check

,minute,player.name,x,y
0,0,Mario Mandžukić,61.0,41.0
1,0,Marcelo Brozović,49.0,41.0
2,0,Luka Modrić,43.0,56.0
3,0,Šime Vrsaljko,38.0,78.0
4,0,Danijel Subašić,9.0,49.0


In [4]:
# List of players that played this match
players = event['player.name'].dropna().unique()
print(players)

<StringArray>
[                            'Mario Mandžukić',
                            'Marcelo Brozović',
                              'Olivier Giroud',
                                 'Luka Modrić',
                               'Šime Vrsaljko',
                              'Blaise Matuidi',
                             'Danijel Subašić',
                              'Raphaël Varane',
                                'Ivan Perišić',
                                'Ivan Rakitić',
                                'Ivan Strinić',
                                  'Paul Pogba',
                                'Domagoj Vida',
                                'Dejan Lovren',
                             'Benjamin Pavard',
                               'N''Golo Kanté',
                           'Antoine Griezmann',
                        'Kylian Mbappé Lottin',
                                 'Hugo Lloris',
                          'Samuel Yves Umtiti',
                          

In [5]:
print(events["duel.outcome.name"].unique())
print(events["interception.outcome.name"].unique())

<StringArray>
[nan, 'Lost Out', 'Success In Play', 'Won', 'Success Out']
Length: 5, dtype: str
<StringArray>
[nan, 'Won', 'Success In Play', 'Lost Out', 'Lost In Play']
Length: 5, dtype: str


In [6]:
# Import needed libraries
import sys
sys.path.append('../src')
import pandas as pd

from etl.extract import get_matches, get_events
from etl.transform import build_all_players_stats
from etl.load import create_tables, save_to_db

# Recreate tables with new columns
create_tables(reset=True)

matches = get_matches(43, 3)    # WorldCup 2018

all_stats = []

for match_id in matches['match_id']:
    events = get_events(match_id)
    all_stats.append(build_all_players_stats(events, match_id))
    print("Match:", match_id, "is ready!")

league_df = pd.concat(all_stats, ignore_index=True)
save_to_db(league_df, "Match_Metrics")

print("Rows and Columns: ", league_df.shape)
# print("Positions: ", league_df['position'].unique())
# league_df.head(10)
league_df[["player_name", "position", "total_passes", "minutes_played"]].head(15)

Tables created successfully.
Match: 8650 is ready!
Match: 7584 is ready!
Match: 7554 is ready!
Match: 7539 is ready!
Match: 7550 is ready!
Match: 7538 is ready!
Match: 7534 is ready!
Match: 7543 is ready!
Match: 7544 is ready!
Match: 7546 is ready!
Match: 7545 is ready!
Match: 7585 is ready!
Match: 7569 is ready!
Match: 7572 is ready!
Match: 7561 is ready!
Match: 7540 is ready!
Match: 7531 is ready!
Match: 7570 is ready!
Match: 7586 is ready!
Match: 7557 is ready!
Match: 7542 is ready!
Match: 8656 is ready!
Match: 7548 is ready!
Match: 7578 is ready!
Match: 7553 is ready!
Match: 7536 is ready!
Match: 7555 is ready!
Match: 7576 is ready!
Match: 7565 is ready!
Match: 7551 is ready!
Match: 7537 is ready!
Match: 7580 is ready!
Match: 7581 is ready!
Match: 7549 is ready!
Match: 7529 is ready!
Match: 7562 is ready!
Match: 7571 is ready!
Match: 7568 is ready!
Match: 7530 is ready!
Match: 7558 is ready!
Match: 7583 is ready!
Match: 7547 is ready!
Match: 7535 is ready!
Match: 8649 is ready!
Mat

,player_name,position,total_passes,minutes_played
0,Youri Tielemans,Left Center Midfield,1,9
1,Fernando Luiz Roza,Center Midfield,66,95
2,Jan Vertonghen,Left Back,47,95
3,Kevin De Bruyne,Secondary Striker,42,95
4,Vincent Kompany,Left Center Back,44,95
5,Thomas Meunier,Right Back,29,95
6,Gabriel Fernando de Jesus,Center Forward,10,57
7,Romelu Lukaku Menama,Right Wing,13,86
8,Thiago Emiliano da Silva,Right Center Back,54,95
9,Marouane Fellaini-Bakkioui,Right Center Midfield,35,95


In [7]:
# Import needed libraries
import sys
sys.path.append("../src")
from etl.extract import get_events

events = get_events(8650)

# All the events types that exist in the match
print(events["type.name"].unique())

# How many events have the word "Starting" in their name?
print(events[events["type.name"].str.contains("Starting", na = False)].shape)

# Does the lineup column exist?
print("tactics.lineup" in events.columns)

starting_xi = events[events["type.name"] == "Starting XI"]

<StringArray>
[      'Starting XI',        'Half Start',              'Pass',
     'Ball Receipt*',             'Carry',          'Pressure',
      'Dispossessed',              'Duel',             'Block',
     'Ball Recovery',              'Shot',       'Goal Keeper',
        'Miscontrol',           'Dribble',    'Foul Committed',
          'Foul Won',         'Camera On',         'Clearance',
     'Dribbled Past',             'Error',      'Own Goal For',
  'Own Goal Against',        'Camera off',      'Interception',
        'Player Off',         'Player On',          'Half End',
      'Substitution',    'Tactical Shift', 'Referee Ball-Drop',
   'Injury Stoppage']
Length: 31, dtype: str
(2, 104)
True


In [8]:
lineup = events[events["type.name"] == "Starting XI"]["tactics.lineup"].iloc[0]
print(len(lineup))
print(lineup[0])

11
{'player': {'id': 5547, 'name': 'Alisson Ramsés Becker'}, 'position': {'id': 1, 'name': 'Goalkeeper'}, 'jersey_number': 1}


In [9]:
from etl.transform import get_minutes_played

minutes = get_minutes_played(events)
print(len(minutes))
print(minutes)

27
{5547: 95, 5838: 95, 3295: 95, 5551: 95, 5552: 95, 5542: 72, 3054: 95, 3501: 95, 3472: 45, 4320: 95, 3202: 57, 3509: 95, 3176: 95, 20005: 95, 3101: 95, 3077: 95, 3296: 95, 5642: 95, 4831: 82, 3289: 86, 3621: 95, 3089: 95, 3535: 50, 5837: 38, 5553: 23, 6332: 13, 2954: 9}


In [11]:
# Import needed Libraries
import sys
sys.path.append("../src")
from etl.extract import get_events
from etl.transform import build_all_players_stats

events = get_events(8658)       # Worldcup 2018 final, France vs Croatia
stats = build_all_players_stats(events, 8658)

print("Goals in the match:", stats["total_goals"].sum())
stats[stats["total_goals"] > 0][["player_name", "total_shots", "total_goals", "total_assists", "total_xg"]]

Goals in the match: 29


,player_name,total_shots,total_goals,total_assists,total_xg
1,Kylian Mbappé Lottin,2,1,0,0.063
4,Dejan Lovren,2,1,0,0.102
5,Olivier Giroud,1,1,0,0.070
6,N''Golo Kanté,0,1,0,0.000
7,Blaise Matuidi,0,2,0,0.000
8,Ivan Strinić,0,1,0,0.000
9,Marko Pjaca,0,1,0,0.000
14,Domagoj Vida,2,1,1,0.269
15,Marcelo Brozović,0,2,0,0.000
16,Ivan Rakitić,3,2,0,0.119
